In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/sentimentt/complaints_nlp.csv


In [4]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np
from tqdm import tqdm

# -----------------------------
# Load your raw dataset
# -----------------------------
df = pd.read_csv("/kaggle/input/sentimentt/complaints_nlp.csv")

text_col = "Consumer complaint narrative"   # update if needed
df = df.dropna(subset=[text_col])

# -----------------------------
# Load sentiment model
# -----------------------------
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# -----------------------------
# Enable GPU
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print("Using device:", device)

labels = ["negative", "neutral", "positive"]

# -----------------------------
# Batch Prediction Function
# -----------------------------
def predict_sentiment_batch(texts, batch_size=64):
    results = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        ).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits

        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)

        # Convert to sentiment label
        sentiments = [labels[p] for p in preds]
        results.extend(sentiments)

    return results

# -----------------------------
# 🔥 RUN SENTIMENT LABELING FAST
# -----------------------------
df["sentiment"] = predict_sentiment_batch(df[text_col].tolist(), batch_size=64)

df.head()


/tmp/ipykernel_47/3563981305.py:10: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/kaggle/input/sentimentt/complaints_nlp.csv")
Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Using device: cuda


100%|██████████| 5994/5994 [1:34:48<00:00,  1.05it/s]


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,sentiment
0,03/23/2019,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Account information incorrect,The Summer of XX/XX/2018 I was denied a mortga...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",IL,NaN,NaN,Consent provided,Web,03/23/2019,Closed with explanation,Yes,NaN,3189109,negative
1,03/22/2019,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Account information incorrect,There are many mistakes appear in my report wi...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",VA,220XX,NaN,Consent provided,Web,03/22/2019,Closed with explanation,Yes,NaN,3187982,negative
2,03/22/2019,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Account information incorrect,There are many mistakes appear in my report wi...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",TX,770XX,NaN,Consent provided,Web,03/22/2019,Closed with explanation,Yes,NaN,3187954,negative
3,03/22/2019,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Account information incorrect,There are many mistakes appear in my report wi...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",TX,787XX,NaN,Consent provided,Web,03/22/2019,Closed with explanation,Yes,NaN,3188091,negative
4,03/22/2019,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Account information incorrect,There are many mistakes appear in my report wi...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",CA,951XX,NaN,Consent provided,Web,03/22/2019,Closed with explanation,Yes,NaN,3188119,negative


In [6]:
df["text"] = df["Consumer complaint narrative"]
df["label"] = df["sentiment"]   # values = negative / neutral / positive


In [8]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
import torch

# ---------------------------------------
# LOAD LABELED DATA
# ---------------------------------------
df = df.dropna(subset=["text", "label"])

# ---------------------------------------
# ENCODE LABELS
# ---------------------------------------
labels = sorted(df["label"].unique())  # ['negative', 'neutral', 'positive']
label2id = {v: i for i, v in enumerate(labels)}
id2label = {i: v for v, i in label2id.items()}

df["label_id"] = df["label"].map(label2id)

# ---------------------------------------
# SPLIT DATA
# ---------------------------------------
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df["label_id"],
    random_state=42
)

# ---------------------------------------
# LOAD TOKENIZER
# ---------------------------------------
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

# Convert to HuggingFace Datasets
train_dataset = Dataset.from_pandas(train_df[["text", "label_id"]])
val_dataset = Dataset.from_pandas(val_df[["text", "label_id"]])

# Tokenize
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset   = val_dataset.map(tokenize, batched=True)

# Rename label column
train_dataset = train_dataset.rename_column("label_id", "labels")
val_dataset = val_dataset.rename_column("label_id", "labels")

train_dataset.set_format("torch")
val_dataset.set_format("torch")

# ---------------------------------------
# LOAD MODEL
# ---------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

# Move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"🔥 Training on device: {device}")

# ---------------------------------------
# TRAINING ARGUMENTS
# ---------------------------------------
args = TrainingArguments(
    output_dir="/kaggle/working/sentiment_model",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,                         # Fast GPU training
    logging_steps=50,
    report_to="none"
)

# ---------------------------------------
# TRAINER
# ---------------------------------------
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

print("🚀 Training started...")
trainer.train()

# ---------------------------------------
# SAVE MODEL
# ---------------------------------------
model.save_pretrained("/kaggle/working/sentiment_model")
tokenizer.save_pretrained("/kaggle/working/sentiment_model")

print("✔ Sentiment model saved successfully!")


Map:   0%|          | 0/345207 [00:00<?, ? examples/s]

Map:   0%|          | 0/38357 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🔥 Training on device: cuda
🚀 Training started...


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,0.209600,0.223058
2,0.171900,0.203476
3,0.121700,0.208866


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


✔ Sentiment model saved successfully!


In [9]:
from transformers import pipeline

clf = pipeline(
    "text-classification",
    model="/kaggle/working/sentiment_model",
    tokenizer="/kaggle/working/sentiment_model",
    device=0
)

print(clf("I am very disappointed with customer service"))
print(clf("Everything was smooth and easy"))


Device set to use cuda:0


[{'label': 'negative', 'score': 0.9998756647109985}]
[{'label': 'positive', 'score': 0.9844422936439514}]


In [10]:
import shutil

shutil.make_archive(
    "/kaggle/working/sentiment_model_zip",
    'zip',
    "/kaggle/working/sentiment_model"
)

print("✔ Zip created!")


✔ Zip created!


In [11]:
import shutil
shutil.rmtree("/kaggle/working/sentiment_model", ignore_errors=True)


In [12]:
import os
os.makedirs("/kaggle/working/sentiment_model", exist_ok=True)


In [13]:
model.save_pretrained("/kaggle/working/sentiment_model")
tokenizer.save_pretrained("/kaggle/working/sentiment_model")


('/kaggle/working/sentiment_model/tokenizer_config.json',
 '/kaggle/working/sentiment_model/special_tokens_map.json',
 '/kaggle/working/sentiment_model/vocab.txt',
 '/kaggle/working/sentiment_model/added_tokens.json',
 '/kaggle/working/sentiment_model/tokenizer.json')

In [14]:
shutil.make_archive(
    "/kaggle/working/sentiment_model_clean",
    'zip',
    "/kaggle/working/sentiment_model"
)


'/kaggle/working/sentiment_model_clean.zip'

In [16]:
from IPython.display import FileLink
FileLink('/kaggle/working/sentiment_model_clean.zip')



/kaggle/working/sentiment_model_clean.zip

In [17]:
!ls -lh /kaggle/working/sentiment_model


total 257M
-rw-r--r-- 1 root root  727 Dec  5 22:55 config.json
-rw-r--r-- 1 root root 256M Dec  5 22:55 model.safetensors
-rw-r--r-- 1 root root  125 Dec  5 22:55 special_tokens_map.json
-rw-r--r-- 1 root root 1.2K Dec  5 22:55 tokenizer_config.json
-rw-r--r-- 1 root root 695K Dec  5 22:55 tokenizer.json
-rw-r--r-- 1 root root 227K Dec  5 22:55 vocab.txt


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [18]:
from IPython.display import FileLink

FileLink("sentiment_model_clean.zip")


/kaggle/working/sentiment_model_clean.zip